<a href="https://colab.research.google.com/github/Rhea-Shah23/indiancine.ma-scrape-curate/blob/main/MACS207_Indiancine_ma_Scrape_and_Curate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!playwright install-deps

Installing dependencies...
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,986 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,862 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all

In [7]:
!pip install playwright

In [8]:
!playwright install

In [9]:
!pip install cinemagoer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.2/297.2 kB 5.4 MB/s eta 0:00:00


In [11]:
# Use Playwright to scrape dynamic content from Indian Cine.ma
import asyncio
from playwright.async_api import async_playwright
import pandas as pd
from imdb import Cinemagoer as IMDb
import re

# Install and apply nest_asyncio to allow nested event loops in Jupyter Notebook
!pip install nest_asyncio
import nest_asyncio
nest_asyncio.apply()

# Initialize IMDb API
ia = IMDb()

# Function to scrape data using Playwright
async def scrape_with_playwright():
    movies = []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)  # Set headless=False to see the browser
        page = await browser.new_page()
        await page.goto("https://indiancine.ma/grid/year")

        # Wait for the page to load the movie elements
        await page.wait_for_selector('.OxElement')

        await asyncio.sleep(5)

        # Extract movie elements
        movie_elements = await page.query_selector_all('.OxText')
        print(movie_elements)
        for movie in movie_elements:
            try:
                title_element = await movie.query_selector('.OxTarget')
                if title_element:
                    full_text = await title_element.text_content()
                    match = re.match(r"^(.*?)(\d{4})$", full_text.strip())
                    if match:
                        title = match.group(1).strip()
                        year = match.group(2).strip()
                    else:
                        title = full_text.strip()
                        year = "N/A"

                    # Clean title before IMDb search - remove director name in parentheses
                    search_title = re.sub(r'\s*\(.*?\)\s*$', '', title).strip()

                    # Search for the movie on IMDb, filtering by year for accuracy
                    search_results = ia.search_movie(title)
                    imdb_id = 'N/A'
                    if search_results:
                      if year != "N/A":
                        for result in search_results[:5]:
                          result_year = str(result.get('year', ''))
                          result_title = result.get('title', '').lower().strip()
                          if result_title == title.lower().strip() and abs(int(result_year or 0) - int(year)) <= 1:
                            imdb_id = result.movieID
                            break
                      # Fall back to first result if no year match found
                      if imdb_id == 'N/A':
                        imdb_id = search_results[0].movieID

                    movies.append({"Title": title, "Year": year, "IMDbID": imdb_id})
            except Exception as e:
                print(f"Error extracting data for a movie: {e}")

        await browser.close()

    # Save the extracted data into a CSV file
    df = pd.DataFrame(movies)
    display(df)
    # output_file = "indian_cinema_movies_with_imdb_playwright.csv"
    # df.to_csv(output_file, index=False)
    # print(f"Data saved to {output_file}")

# Run the function
await scrape_with_playwright()

[<JSHandle preview=JSHandle@<div class="OxText">…</div>>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHandle@node>, <JSHandle preview=JSHand

,Title,Year,IMDbID
0,Cocoanut Fair (Foreign photographer who possib...,1897,N/A
1,Our Indian Empire,1897,N/A
2,Dancing Scenes from the Flower of Persia (Hira...,1898,N/A
3,A Panorama of Indian Scenes and Procession (Pr...,1898,N/A
4,Man and Monkey (H.S. Bhatavdekar),1899,N/A
...,...,...,...
121,Congress Session in Bombay (Baburao Painter),1919,N/A
122,Kabir Kamal (S.N. Patankar),1919,N/A
123,Kacha Devayani (S.N. Patankar),1919,N/A
124,Narasinh Avatar (S.N. Patankar),1919,N/A
